In [1]:
from sdv.evaluation.single_table import run_diagnostic, evaluate_quality, get_column_plot
from sdv.metadata import Metadata
import pandas as pd
import os

In [2]:
# Constants
REAL_DATA_PATH = "../../Datasets/csv/UAVATTACK/spoofing-merged-gps-only.csv"
METADATA_FILE = "../../Synthetic_Data_Generation/UAV Synthetic Data Generation/SDV/spoofing_metadata.json"
COLUMN_TO_PLOT = "lat_y"  # Change this to the column you want to plot

In [3]:
# Load real data
real_data = pd.read_csv(REAL_DATA_PATH)

In [4]:
# Data Preprocessing
if 'timestamp' in real_data.columns:
    real_data = real_data.drop(columns=['timestamp'])

# Clean up column names
real_data.columns = real_data.columns.str.replace('[\\[\\]<>]', '_', regex=True)

# Encode 'label'
if 'label' in real_data.columns and (real_data['label'].dtype == object or real_data['label'].dtype.name == 'category'):
    from sklearn.preprocessing import LabelEncoder

    le = LabelEncoder()
    real_data['label'] = le.fit_transform(real_data['label'])

In [3]:
# Load or create metadata
if os.path.exists(METADATA_FILE):
    metadata = Metadata.load_from_json(METADATA_FILE)
    print(f"Loaded metadata from {METADATA_FILE}")
else:
    metadata = Metadata()
    metadata.detect_table_from_dataframe(
        data=real_data,
        table_name='spoofing_data'
    )
    metadata.save_to_json(METADATA_FILE)
    print(f"Created and saved new metadata to {METADATA_FILE}")

Loaded metadata from ../../experiments/UAVATTACK/Synthetic Data Generation/SDV/spoofing_metadata.json


C:\Users\antho\miniforge3\envs\CTGAN\Lib\site-packages\sdv\metadata\metadata.py:37: UserWarning: You are loading an older SingleTableMetadata object. This will be converted into the new Metadata object with a placeholder table name ('table'). Please save this new object for future usage.
  warnings.warn(


In [6]:
# Synthesizer
synthesizers = [
    {
        'name': 'CTGAN',
        'output': "../../Datasets/csv/UAVATTACK/spoofing-CTGAN-synthetic.csv"
    },
    {
        'name': 'GaussianCopula',
        'output': "../../Datasets/csv/UAVATTACK/spoofing-GaussianCopula-synthetic.csv"
    },
    {
        'name': 'TVAE',
        'output': "../../Datasets/csv/UAVATTACK/spoofing-TVAE-synthetic.csv"
    },
    {
        'name': 'CGAN',
        'output': "../../Datasets/csv/UAVATTACK/syntheticSpoofing_cgan_data.csv"
    },
    {
        'name': 'GAN',
        'output': "../../Datasets/csv/UAVATTACK/syntheticSpoofing_gan_data.csv"
    }

]

In [7]:
for synth in synthesizers:
    print(f"--- {synth['name']} ---")
    synthetic_data = pd.read_csv(synth['output'])

    diagnostic = run_diagnostic(real_data, synthetic_data, metadata)

    quality_report = evaluate_quality(real_data, synthetic_data, metadata)

--- CTGAN ---
Generating report ...

(1/2) Evaluating Data Validity: |██████████| 83/83 [00:00<00:00, 1434.02it/s]|
Data Validity Score: 100.0%

(2/2) Evaluating Data Structure: |██████████| 1/1 [00:00<00:00, 284.63it/s]|
Data Structure Score: 97.62%

Overall Score (Average): 98.81%

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 83/83 [00:09<00:00,  8.34it/s]|
Column Shapes Score: 83.84%

(2/2) Evaluating Column Pair Trends: |██████████| 3403/3403 [00:09<00:00, 344.08it/s]|
Column Pair Trends Score: 89.83%

Overall Score (Average): 86.83%

--- GaussianCopula ---
Generating report ...

(1/2) Evaluating Data Validity: |██████████| 83/83 [00:00<00:00, 1728.77it/s]|
Data Validity Score: 100.0%

(2/2) Evaluating Data Structure: |██████████| 1/1 [00:00<00:00, 397.79it/s]|
Data Structure Score: 97.62%

Overall Score (Average): 98.81%

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 83/83 [00:13<00:00,  6.20it/s]|
Column Shapes Score: 77.95%

(2/2) Eva

In [8]:
for synth in synthesizers:
    print(f"--- {synth['name']} ---")
    synthetic_data = pd.read_csv(synth['output'])
    fig = get_column_plot(
        real_data=real_data,
        synthetic_data=synthetic_data,
        metadata=metadata,
        column_name=COLUMN_TO_PLOT
    )
    fig.show()

--- CTGAN ---


--- GaussianCopula ---


--- TVAE ---


--- CGAN ---


--- GAN ---
